[⬅ Back to the examples](https://anywidgetinstruments.github.io/anywidget-instruments-industrial/try/)

# Operator station

A filling line follows the machine state model. **Run all the cells**, then
press **Reset** and **Start** on the state diagram. Acting states (dashed)
complete after two seconds. The stack light, the temperature loop, the
annunciator and the alarm list follow the line; after 20 s in Execute the
capper jams and the line suspends: acknowledge, then **Unsuspend**.

The simulation runs as an `asyncio` task (the browser has no threads), so the
notebook stays responsive while it runs.

**References:** ISA-TR88.00.02 (PackML machine states), ISA-18.1
(annunciator sequences), ISA-18.2 / IEC 62682 (alarm list, shelving),
ISA-101 (PID faceplate), IEC 60073 (stack light colors). These standards
inspired the widgets; the library does not claim conformity with them
([Standards and
references](https://anywidgetinstruments.github.io/anywidget-instruments-industrial/standards/)).

> **The code is hidden.** To see a cell's code, click the `⋯` bar above its result; to see all of it, choose **View ▸ Expand All Code**.

In [ ]:
# JupyterLite: install the package (bundled with this site) in the browser kernel
%pip install -q anywidget-instruments-industrial

In [ ]:
import asyncio

import anywidget_instruments_industrial as ai

machine = ai.StateMachine(label="Line state", size=(720, 380))
light = ai.StackLight(
    tiers=["red", "amber", "green"], labels=["Fault", "Attention", "Producing"], label="Stack light"
)
heater = ai.PIDFaceplate(
    tag="TIC-101",
    unit="°C",
    pv_max=90,
    hi=70,
    confirm_delta=10,
    label="Product temperature",
    controller=ai.PID(kp=4.0, ti=40.0, sp=55.0),
)
count = ai.SevenSegment(0, digits=6, decimals=0, label="Bottles", size=(180, 60))
annunciator = ai.Annunciator(
    [("TAH-101", "Temp high", "red"), ("XA-401", "Capper jam", "amber")],
    sequence="R",
    first_out=True,
    columns=2,
    label="Annunciator",
    size=(260, 110),
)
alarms = ai.AlarmList(label="Alarm list", size=(600, 170))
ai.Panel([machine, light, heater, ai.Panel([count, annunciator], columns=1), alarms], columns=2)

In [ ]:
ai.theme_switch()  # light / system / dark theme (STYLE-008)

In [ ]:
PATTERN = {
    "Execute": {"green": "on"},
    "Starting": {"green": "blink"},
    "Held": {"amber": "on"},
    "Suspended": {"amber": "blink"},
    "Aborted": {"red": "blink"},
}
sim = {"temp": 25.0, "since": 0, "last": machine.value, "producing": 0, "jammed": False}


def tick():
    """Advance the line by one second."""
    if machine.value != sim["last"]:
        sim["last"], sim["since"] = machine.value, 0
    sim["since"] += 1
    if machine.is_acting and sim["since"] >= 2:
        machine.state_complete()
    pattern = PATTERN.get(machine.value, {} if machine.value == "Stopped" else {"amber": "on"})
    light.value = [pattern.get(c, "off") for c in light.tiers]
    op = heater.step(sim["temp"], 1.0)
    sim["temp"] += (-(sim["temp"] - 20.0) + 0.9 * op) / 60.0
    if machine.value == "Execute":
        sim["producing"] += 1
        count.value = count.value + 4
        if sim["producing"] >= 20 and not sim["jammed"]:
            sim["jammed"] = True
            annunciator.set("XA-401", True)
            alarms.raise_alarm("XA-401", "Capper jam", source="XA-401", priority="medium")
            machine.command("Suspend")
    elif machine.value == "Unsuspending" and sim["jammed"]:
        sim["jammed"] = None  # jam cleared by the operator's Unsuspend
        annunciator.set("XA-401", False)
        alarms.clear_alarm("XA-401")
    hot = sim["temp"] > heater.hi
    if hot != (annunciator.state_of("TAH-101") != "normal"):
        annunciator.set("TAH-101", hot)
        if hot:
            alarms.raise_alarm("TAH-101", "Product temperature high", source="TIC-101")
        else:
            alarms.clear_alarm("TAH-101")


async def run(seconds=1800):
    for _ in range(seconds):
        tick()
        await asyncio.sleep(1.0)


task = asyncio.ensure_future(run())  # noqa: RUF006 - stop it with task.cancel()